# 📏 Multi-Layer Normative Limits & Compliance (SQL)

**Author:** Gabriella Marín  
**Project:** Multi-Layer Water Quality Risk & Regulatory Analytics System   
**Phase:** Phase 3 – Normative Compliance (Multi-Layer)

**Objective:**  
Store Colombian regulatory limits (Resolution 2115/2007 and Resolution 0631/2015) and an EPA benchmark
in a relational SQLite database, then compute compliance indicators per sample and parameter.

## 1. Imports, paths and data base conections

In [1]:
import sqlite3
import pandas as pd
import numpy as np
from pathlib import Path

# Project structure
PROJECT_DIR = Path(r"D:\Documents\Portfolio\01-water-quality-normative")

DATA_DIR = PROJECT_DIR / "data"
NORM_DIR = PROJECT_DIR / "norms"

DATA_DIR.mkdir(exist_ok=True)
NORM_DIR.mkdir(exist_ok=True)

DB_PATH = DATA_DIR / "water_quality_analysis.db"
conn = sqlite3.connect(DB_PATH)

# Load parameter catalog
dim_parameter = pd.read_sql_query("SELECT * FROM dim_parameter;",conn)
display(dim_parameter)

parameter_map = dict(zip(dim_parameter["parameter_std"], dim_parameter["parameter_id"]))

,parameter_id,parameter_std,canonical_unit
0,1,Ammoniacal Nitrogen,mg N-NH3-/L
1,2,BOD5,mg O2/L
2,3,COD,mg O2/L
3,4,Chloride,mg Cl-/L
4,5,Dissolved Oxygen,mg O2/L
5,6,Electrical Conductivity,µS/cm
6,7,Nitrate,mg N-NO3-/L
7,8,Nitrite,mg/L
8,9,Sulfate,mg SO4-2/L
9,10,Temperature,°C


## 2. Normative Configuration File
Regulatory limits are stored externally in a CSV file to ensure transparency
and maintainability. If the file does not exist, a minimal version is created.

In [ ]:
# Normative limits CSV (bootstrap)
LIMITS_CSV = NORM_DIR / "norm_limits_master.csv"

csv_text = """standard_code,parameter_std,limit_type,limit_min,limit_max,limit_unit,basis,notes
CO_2115,pH,range,6.5,9.0,pH units,,Colombia drinking water guideline
CO_2115,Turbidity,max,,2.0,NTU,,Colombia drinking water guideline
EPA,pH,range,6.5,8.5,pH units,,EPA benchmark
EPA,Turbidity,max,,5.0,NTU,,EPA benchmark
CO_0631,COD,max,,150.0,mg/L,,Discharge standard reference
CO_0631,BOD5,max,,50.0,mg/L,,Discharge standard reference
CO_0631,Total Suspended Solids,max,,100.0,mg/L,,Discharge standard reference
CO_0631,Temperature,max,,35.0,°C,,Discharge standard reference
"""

if not LIMITS_CSV.exists():
    LIMITS_CSV.write_text(csv_text, encoding="utf-8")
    print("Normative limits CSV created.")
else:
    print("Normative limits CSV already exists.")

limits_df = pd.read_csv(LIMITS_CSV)
display(limits_df)

Normative limits CSV already exists.


,standard_code,parameter_std,limit_type,limit_min,limit_max,limit_unit,basis,notes
0,CO_2115,pH,range,6.5,9.0,pH units,NaN,Colombia drinking water guideline
1,CO_2115,Turbidity,max,NaN,2.0,NTU,NaN,Colombia drinking water guideline
2,EPA,pH,range,6.5,8.5,pH units,NaN,EPA benchmark
3,EPA,Turbidity,max,NaN,5.0,NTU,NaN,EPA benchmark
4,CO_0631,COD,max,NaN,150.0,mg/L,NaN,Discharge standard reference
5,CO_0631,BOD5,max,NaN,50.0,mg/L,NaN,Discharge standard reference
6,CO_0631,Total Suspended Solids,max,NaN,100.0,mg/L,NaN,Discharge standard reference
7,CO_0631,Temperature,max,NaN,35.0,°C,NaN,Discharge standard reference


## 3. Create Normative Tables and Load Limits

In [ ]:
# Create normative tables
cur = conn.cursor()

cur.executescript("""
DROP TABLE IF EXISTS dim_layer;
DROP TABLE IF EXISTS dim_standard;
DROP TABLE IF EXISTS fact_norm_limit;
DROP TABLE IF EXISTS fact_compliance;

CREATE TABLE dim_layer (
    layer_id INTEGER PRIMARY KEY,
    layer_code TEXT UNIQUE,
    layer_name TEXT);

CREATE TABLE dim_standard (
    standard_id INTEGER PRIMARY KEY,
    standard_code TEXT UNIQUE,
    standard_name TEXT,
    layer_id INTEGER);

CREATE TABLE fact_norm_limit (
    standard_id INTEGER,
    parameter_id INTEGER,
    limit_min REAL,
    limit_max REAL,
    limit_type TEXT,
    limit_unit TEXT,
    basis TEXT,
    notes TEXT,
    PRIMARY KEY (standard_id, parameter_id));

CREATE TABLE fact_compliance (
    standard_id INTEGER,
    CODIGO__MUESTRA INTEGER,
    parameter_id INTEGER,
    value_num REAL,
    compliant INTEGER,
    deviation REAL,
    PRIMARY KEY (standard_id, CODIGO__MUESTRA, parameter_id));
""")

# Insert layers
layers = [(1, "SANITARY", "Sanitary / Drinking-water reference"),
            (2, "DISCHARGE", "Environmental / Discharge pressure")]
cur.executemany("INSERT INTO dim_layer VALUES (?, ?, ?);", layers)

# Insert standards
standards = [ (1, "CO_2115", "Colombia Resolution 2115/2007", 1),
            (2, "EPA", "US EPA Drinking Water (Benchmark)", 1),
            (3, "CO_0631", "Colombia Resolution 0631/2015", 2)]
cur.executemany("INSERT INTO dim_standard VALUES (?, ?, ?, ?);", standards)

conn.commit()

# Insert limits
std_map = pd.read_sql_query("SELECT standard_id, standard_code FROM dim_standard;",conn)
std_map = dict(zip(std_map["standard_code"], std_map["standard_id"]))

limits_df["standard_id"] = limits_df["standard_code"].map(std_map)
limits_df["parameter_id"] = limits_df["parameter_std"].map(parameter_map)

insert_df = limits_df[
    ["standard_id", "parameter_id", "limit_min", "limit_max", "limit_type", "limit_unit", "basis", "notes"]]

cur.executemany("INSERT OR REPLACE INTO fact_norm_limit VALUES (?, ?, ?, ?, ?, ?, ?, ?);",
    insert_df.itertuples(index=False, name=None))

conn.commit()

display(pd.read_sql_query("SELECT * FROM fact_norm_limit;",conn))

,standard_id,parameter_id,limit_min,limit_max,limit_type,limit_unit,basis,notes
0,1,13,6.5,9.0,range,pH units,None,Colombia drinking water guideline
1,1,12,NaN,2.0,max,NTU,None,Colombia drinking water guideline
2,2,13,6.5,8.5,range,pH units,None,EPA benchmark
3,2,12,NaN,5.0,max,NTU,None,EPA benchmark
4,3,3,NaN,150.0,max,mg/L,None,Discharge standard reference
5,3,2,NaN,50.0,max,mg/L,None,Discharge standard reference
6,3,11,NaN,100.0,max,mg/L,None,Discharge standard reference
7,3,10,NaN,35.0,max,°C,None,Discharge standard reference


## 4. Compute Compliance Indicators

In [ ]:
# Load measurements and limits
measurements = pd.read_sql_query("""
SELECT CODIGO__MUESTRA, parameter_id, value_num
FROM fact_measurement
WHERE value_num IS NOT NULL;
""", conn)

norms = pd.read_sql_query("""
SELECT standard_id, parameter_id, limit_min, limit_max, limit_type
FROM fact_norm_limit;
""", conn)

dfc = measurements.merge(norms, on="parameter_id", how="inner")

# Compliance logic
def evaluate(row):
    v = row["value_num"]
    lo = row["limit_min"]
    hi = row["limit_max"]
    t  = row["limit_type"]

    if t == "range":
        ok = (v >= lo) & (v <= hi)
        dev = 0 if ok else (lo - v if v < lo else v - hi)
    elif t == "max":
        ok = v <= hi
        dev = 0 if ok else v - hi
    else:
        return pd.Series([None, None])

    return pd.Series([int(ok), float(dev)])

dfc[["compliant", "deviation"]] = dfc.apply(evaluate, axis=1)

cur.executemany("""
INSERT OR REPLACE INTO fact_compliance
VALUES (?, ?, ?, ?, ?, ?);
""", dfc[[
    "standard_id", "CODIGO__MUESTRA", "parameter_id",
    "value_num", "compliant", "deviation"
]].itertuples(index=False, name=None))

conn.commit()

display(pd.read_sql_query(
"SELECT * FROM fact_compliance LIMIT 20;", conn))

,standard_id,CODIGO__MUESTRA,parameter_id,value_num,compliant,deviation
0,3,14615,2,2.60,1,0.00
1,3,20148,2,2.00,1,0.00
2,3,22820,2,2.00,1,0.00
3,3,25379,2,5.00,1,0.00
4,3,14615,3,63.00,1,0.00
5,3,20148,3,11.00,1,0.00
6,3,22820,3,10.00,1,0.00
7,3,25379,3,19.00,1,0.00
8,1,14615,13,7.69,1,0.00
9,2,14615,13,7.69,1,0.00


## Phase 3 Summary

In this phase, regulatory limits were implemented using a multi-layer framework
that separates drinking-water standards and discharge regulations.

Compliance was evaluated at the parameter level for each sample and regulatory
standard, generating traceable compliance and deviation indicators.

At this stage, most evaluated measurements appear compliant under the selected
limits. This outcome reflects the parameter-specific and partial nature of
regulatory evaluation, rather than a final classification of water quality.

The resulting compliance table establishes a robust foundation for continuous
risk scoring, cross-standard comparison, and advanced analytical modeling in
subsequent phases.